In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install owlready2

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd
import owlready2 as or2
from sklearn.preprocessing import StandardScaler

In [ ]:
onto = or2.get_ontology("/path/to/ontology/mlprov.owx").load()
with onto:
    class description(or2.DataProperty):
        range = [str]

In [ ]:
req_spec = onto["requirement_specification"]
train_data = onto["training_dataset"]
test_data = onto["testing_dataset"]
data_source = onto["data_source"]
method_collection = onto["Method_of_collection"]
data_collector = onto["data_collector"]
preprocessing_step = onto["preprocessing_step"]
exclusion_criteria = onto["exclusion_criteria"]
class_proportion = onto["dataset_class_split"]
onto_model = onto["model"]
performance_metric = onto["performance_metric"]

In [ ]:
req1 = req_spec("req1")
req1.description = ["The model shall predict the class <=50K."]
req2 = req_spec("req2")
req2.description = ["The model shall predict the class >50K."]

training_dataset = train_data("adult_train.csv")
training_dataset.description = ["Location: https://github.com/AnonymousWriter1/BiasProvenance"]
original_test_dataset = test_data("adult_test.csv")
original_test_dataset.description = ["Location: https://github.com/AnonymousWriter1/BiasProvenance"]
sex_test_dataset = test_data("sex_test.csv")
sex_test_dataset.description = ["Location: https://github.com/AnonymousWriter1/BiasProvenance"]
race_test_dataset = test_data("race_test.csv")
race_test_dataset.description = ["Location: https://github.com/AnonymousWriter1/BiasProvenance"]

source = data_source("1994_Census")
method_of_collection = method_collection("Census_survey")
method_of_collection.description = ["Census survey conducted by the US Census Bureau. Year: 1994"]
collector = data_collector("Government_Official")

drop_na = preprocessing_step("Drop_NA")
excluded = exclusion_criteria("NA")
male = class_proportion("Percentage_Male=0.67")
female = class_proportion("Percentage_Female=0.33")
white = class_proportion("Percentage_White=0.85")
black = class_proportion("Percentage_Black=0.09")
api = class_proportion("Percentage_Asian_Pacific_Islander=0.03")
ai = class_proportion("Percentage_American_Indian=0.009")
other = class_proportion("Percentage_Other_Race=0.008")


In [ ]:
csv_file_path = '/path/to/data/adult_train.csv'
df = pd.read_csv(csv_file_path)
df = df.dropna()
df.head()


,Age,Workclass,fnlwgt,Education,Education_Num,Martial_Status,Occupation,Relationship,Race,Sex,Capital_Gain,Capital_Loss,Hours_per_week,Country,Target
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [ ]:
X = df.iloc[:, :-1]   # all rows, all columns except the last
y = df.iloc[:, -1]    # all rows, just the last column

x_encoded = pd.get_dummies(X)
train_feature_columns = x_encoded.columns
y_encoded = pd.get_dummies(y)
y_encoded = y.map({' <=50K': 0, ' >50K': 1})

In [ ]:
scalar = StandardScaler()
x_encoded = scalar.fit_transform(x_encoded)

In [ ]:
clf = LogisticRegression(max_iter=10000, random_state=0).fit(x_encoded, y_encoded)
model = onto_model("clf")

Test on original testing dataset

In [ ]:
df_test = pd.read_csv('/path/to/data/adult_test.csv')
df_test = df_test.dropna()
df_test.head()

x_test = df_test.iloc[:, :-1]
y_test = df_test.iloc[:, -1]
x_test_encoded = pd.get_dummies(x_test)

# Align the columns of the test set with the training set
x_test_encoded = x_test_encoded.reindex(columns=train_feature_columns, fill_value=0)
y_test_encoded = pd.get_dummies(y_test)
y_test_encoded = y_test.map({' <=50K.': 0, ' >50K.': 1})
x_test_encoded = scalar.transform(x_test_encoded)

# Predictions on the test set
y_test_pred = clf.predict(x_test_encoded)

# accuracy on test set
acc_test = accuracy_score(y_test_encoded, y_test_pred) * 100
print(acc_test)
# add accuracy to the provenance graph
orig_acc_test = performance_metric(f"Original_test_accuracy={acc_test}")

# precision on the test set
precision_test = precision_score(y_test_encoded, y_test_pred)
print(precision_test)
# add precision to the provenance graph
orig_precision_test = performance_metric(f"Original_test_precision={precision_test}")

# recall on the test set
recall_test = recall_score(y_test_encoded, y_test_pred)
print(recall_test)
# add recall to the provenance graph
orig_recall_test = performance_metric(f"Original_test_recall={recall_test}")

# f1 score on the test set
f1_test = f1_score(y_test_encoded, y_test_pred)
print(f1_test)
# add f1 score ot the provenance graph
orig_f1_test = performance_metric(f"Original_test_F1_score={f1_test}")

# auc on the test set
auc_test = roc_auc_score(y_test_encoded, y_test_pred)
print(auc_test)
# add auc to the provenance graph
orig_auc_test = performance_metric(f"Original_test_auc={auc_test}")

82.37051792828686
0.8833455612619222
0.3254054054054054
0.4756073474224768
0.655704463266083


Minority sex test set

In [ ]:
df_test_sex = pd.read_csv('/path/to/data/sex_test.csv')
df_test_sex = df_test_sex.dropna()
df_test_sex.head()

x_test_sex = df_test_sex.iloc[:, :-1]
y_test_sex = df_test_sex.iloc[:, -1]
x_test_sex_encoded = pd.get_dummies(x_test_sex)
# Align the columns of the test set with the training set
x_test_sex_encoded = x_test_sex_encoded.reindex(columns=train_feature_columns, fill_value=0)
y_test_sex_encoded = pd.get_dummies(y_test_sex)
y_test_sex_encoded = y_test_sex.map({' <=50K.': 0, ' >50K.': 1})
x_test_sex_encoded = scalar.transform(x_test_sex_encoded)

# predictions on the minority sex test set
y_pred_sex = clf.predict(x_test_sex_encoded)

# accuracy on the minority sex test set
acc_test_sex = accuracy_score(y_test_sex_encoded, y_pred_sex) * 100
print(acc_test_sex)
# add accuracy to the provenance graph
accuracy_test_sex = performance_metric(f"Minority_sex_accuracy={acc_test_sex}")

# precision on the minority sex test set
prec_test_sex = precision_score(y_test_sex_encoded, y_pred_sex)
print(prec_test_sex)
# add precision to the provenance graph
precision_test_sex = performance_metric(f"Minority_sex_precision={prec_test_sex}")

# recall on the minority sex test set
rec_test_sex = recall_score(y_test_sex_encoded, y_pred_sex)
print(rec_test_sex)
# add recall to the provenance graph
recall_test_sex = performance_metric(f"Minority_sex_recall={rec_test_sex}")

# f1 score on the minority sex test set
f1_sex = f1_score(y_test_sex_encoded, y_pred_sex)
print(f1_sex)
# add f1 score to the provenance graph
f1_test_sex = performance_metric(f"Minority_sex_F1_score={f1_sex}")

# auc on the minority sex test set
auc_sex = roc_auc_score(y_test_sex_encoded, y_pred_sex)
print(auc_sex)
# add auc to the provenance graph
auc_test_sex = performance_metric(f"Minority_sex_AUC={auc_sex}")

91.6751475676776
0.8894736842105263
0.30341113105924594
0.4524765729585007
0.6492950972100637


Minority race test

In [ ]:
df_test_race = pd.read_csv('/path/to/data/race_test.csv')
df_test_race = df_test_race.dropna()
df_test_race.head()

x_test_race = df_test_race.iloc[:, :-1]
y_test_race = df_test_race.iloc[:, -1]
x_test_race_encoded = pd.get_dummies(x_test_race)
# Align the columns of the test set with the training set
x_test_race_encoded = x_test_race_encoded.reindex(columns=train_feature_columns, fill_value=0)
y_test_race_encoded = pd.get_dummies(y_test_race)
y_test_race_encoded = y_test_race.map({' <=50K.': 0, ' >50K.': 1})
x_test_race_encoded = scalar.transform(x_test_race_encoded)

# predictions on the minority race test set
y_pred_race = clf.predict(x_test_race_encoded)

# accuracy on the minority race test set
acc_test_race = accuracy_score(y_test_race_encoded, y_pred_race) * 100
print(acc_test_race)
# add accuracy to the provenance graph
accuracy_test_race_val = performance_metric(f"Minority_race_accuracy={acc_test_race}")

# precision on the minority race test set
prec_test_race = precision_score(y_test_race_encoded, y_pred_race)
print(prec_test_race)
# add precision to the provenance graph
precision_test_race = performance_metric(f"Minority_race_precision={prec_test_race}")

# recall on the minority race test set
rec_test_race = recall_score(y_test_race_encoded, y_pred_race)
print(rec_test_race)
# add recall to the provenance graph
recall_test_race = performance_metric(f"Minority_race_recall={rec_test_race}")

# f1 score on the minority race test set
f1_race = f1_score(y_test_race_encoded, y_pred_race)
print(f1_race)
# add f1 score to the provenance graph
f1_test_race = performance_metric(f"Minority_race_F1_score={f1_race}")

# auc on the minority race test set
auc_race = roc_auc_score(y_test_race_encoded, y_pred_race)
print(auc_race)
# add auc to the provenance graph
auc_test_race = performance_metric(f"Minority_race_AUC={auc_race}")

89.04306220095694
0.7019607843137254
0.5391566265060241
0.6098807495741057
0.7479628411255945


Save provenance graph

In [ ]:
onto.save(file="/path/mlprov_selection_bias_instances.owl")